# [🛠️ Introduction to Functions & Aggregation](https://learn.microsoft.com/en-us/training/modules/use-built-functions-transact-sql/1-introduction)

## Manipulating Data with Functions
When retrieving data from tables in a database, raw data isn't always in the exact format you need for your reports or applications. Transact-SQL provides a rich library of **functions** that allow you to manipulate data values directly within your queries. 

Functions can be used to:
- **Format** data (e.g., changing date formats, extracting parts of a string, or changing text casing).
- **Convert** data types (e.g., turning a string into a number, or a date into a string).
- **Aggregate** data (e.g., calculating sums, averages, or counts).
- **Affect output** in various other ways to meet specific business requirements.

## Summarizing Data with Grouping
Beyond manipulating individual values, you often need to look at the "big picture." When analyzing data, you'll frequently want to **group** the results to show aggregations for each specific category. 

For example, instead of just seeing the total sales for the entire company, you might want to see total sales *by product category*, *by region*, or *by salesperson*. Grouping allows you to collapse thousands of individual rows into a concise, meaningful summary.

## How Data Flows Through Manipulation and Aggregation

When you combine functions, grouping, and filtering, the SQL Server engine processes your data through a specific logical pipeline. Understanding this flow is key to writing accurate queries.

```mermaid
graph TD
    A[(Raw Database Tables)] --> B{Apply Functions}
    
    B -->|Scalar Functions| C[Format & Convert Single Values <br> e.g., UPPER, ROUND]
    B -->|Aggregate Functions| D[Calculate Summaries <br> e.g., SUM, AVG, COUNT]
    
    C --> E[Row-by-Row Output]
    D --> F[GROUP BY Clause <br> Create logical buckets]
    
    F --> G{HAVING Clause}
    G -->|Filter out unwanted groups| H([Final Summarized Report])
    
    style A fill:#e1f5fe,stroke:#0288d1,stroke-width:2px
    style C fill:#fff3e0,stroke:#f57c00
    style D fill:#e8f5e9,stroke:#2e7d32
    style G fill:#f3e5f5,stroke:#8e24aa
```

***

## Module Learning Objectives

In this module, you will master the techniques for transforming and summarizing data in T-SQL. By the end of this module, you will be able to:

1. **Categorize built-in functions**: Understand the different types of functions available in T-SQL (Scalar, Aggregate, Ranking) and when to use them.
2. **Use scalar functions**: Apply functions that operate on a single value and return a single value (e.g., string manipulation, date math, mathematical calculations).
3. **Use ranking and rowset functions**: Assign rankings, row numbers, or generate sequences to your result sets (e.g., `ROW_NUMBER()`, `RANK()`).
4. **Use aggregate functions**: Perform calculations across a set of values to return a single summary value (e.g., `SUM`, `AVG`, `COUNT`, `MIN`, `MAX`).
5. **Summarize data with the `GROUP BY` clause**: Group rows that have the same values into summary rows.
6. **Filter groups with the `HAVING` clause**: Apply conditions to filter the grouped results *after* the aggregation has occurred (unlike `WHERE`, which filters *before* aggregation).


# 🗂️ Categorizing Built-in T-SQL Functions

## Overview
Transact-SQL includes a rich library of built-in functions that allow you to manipulate, transform, and analyze data directly within your queries. These functions range from simple data type conversions to complex aggregations and analytical groupings.

To use them effectively, it is helpful to understand how T-SQL categorizes these functions based on their input, output, and how they process data.

***

## The 5 Core Function Categories

Functions in T-SQL are broadly categorized into five distinct types based on how they process rows and what they return.

```mermaid
graph TD
    classDef title fill:#f5f5f5,stroke:#333,stroke-width:3px;
    classDef scalar fill:#e3f2fd,stroke:#1565c0,stroke-width:2px;
    classDef logical fill:#f3e5f5,stroke:#7b1fa2,stroke-width:2px;
    classDef ranking fill:#fff3e0,stroke:#ef6c00,stroke-width:2px;
    classDef rowset fill:#e8f5e9,stroke:#2e7d32,stroke-width:2px;
    classDef agg fill:#ffebee,stroke:#c62828,stroke-width:2px;
    
    A["T-SQL Built-In Functions"]:::title --> B["1. Scalar"]:::scalar
    A --> C["2. Logical"]:::logical
    A --> D["3. Ranking"]:::ranking
    A --> E["4. Rowset"]:::rowset
    A --> F["5. Aggregate"]:::agg
    
    B --> B1["Input: Single row\nOutput: Single value"]
    C --> C1["Input: Multiple values\nOutput: Single output based on logic"]
    D --> D1["Input: Partition/Set of rows\nOutput: Rank/Row number"]
    E --> E1["Input: Data stream/JSON/XML\nOutput: Virtual table for FROM clause"]
    F --> F1["Input: Multiple rows/values\nOutput: Single summarizing value"]
```

***

## Detailed Breakdown of Each Category

| Category | Description | Common Examples |
| :--- | :--- | :--- |
| **Scalar** | Operate on a **single row**, return a **single value**. | `UPPER()`, `LEFT()`, `GETDATE()`, `YEAR()`, `ABS()`, `CAST()` |
| **Logical** | Compare multiple values to determine a **single output**. | `IIF()`, `CHOOSE()`, `COALESCE()` |
| **Ranking** | Operate on a **partition (set) of rows** to assign a rank or number. | `ROW_NUMBER()`, `RANK()`, `DENSE_RANK()`, `NTILE()` |
| **Rowset** | Return a **virtual table** that can be used in a `FROM` clause. | `OPENJSON()`, `OPENXML()`, `CONTAINSTABLE()` |
| **Aggregate** | Take **one or more input values** (usually a column across many rows), return a **single summarizing value**. | `SUM()`, `AVG()`, `COUNT()`, `MIN()`, `MAX()` |

***

**Examples of Each Function Category**
```sql
-- ==========================================
-- 1. SCALAR FUNCTION
-- ==========================================
-- Operates on a single value/row and returns a single value.
-- Example: Converting a string to uppercase.
SELECT UPPER('hello world') AS ScalarExample;


-- ==========================================
-- 2. LOGICAL FUNCTION
-- ==========================================
-- Compares values and returns a single logical output.
-- Example: IIF (Inline If) evaluates a condition.
SELECT IIF(100 > 50, 'Greater', 'Less or Equal') AS LogicalExample;


-- ==========================================
-- 3. RANKING FUNCTION
-- ==========================================
-- Operates on a partition of rows. Requires the OVER() clause.
-- Example: Assigning a row number based on price.
SELECT TOP 5 ProductID, Name, ListPrice,
       ROW_NUMBER() OVER(ORDER BY ListPrice DESC) AS RankingExample
FROM SalesLT.Product;


-- ==========================================
-- 4. ROWSET FUNCTION
-- ==========================================
-- Returns a virtual table. Often used for parsing JSON or XML.
-- Example: OPENJSON parses JSON text into a relational table format.
-- (Conceptual syntax): 
-- SELECT * FROM OPENJSON(@json_variable);


-- ==========================================
-- 5. AGGREGATE FUNCTION
-- ==========================================
-- Takes multiple input values (a whole column) and returns a single summary.
-- Example: Counting the total number of products.
SELECT COUNT(ProductID) AS AggregateExample
FROM SalesLT.Product;
```

***

## Summary

- **Scalar functions** work on a row-by-row basis (1 row in $\rightarrow$ 1 value out).
- **Logical functions** evaluate conditions and return a single result based on that logic.
- **Ranking functions** analyze a set (partition) of rows to assign sequential numbers or ranks.
- **Rowset functions** are unique because they return an entire **table** rather than a single value, allowing them to be used directly in the `FROM` clause.
- **Aggregate functions** collapse multiple rows into a single summarized value (like a total or average).

Understanding these categories helps you choose the right tool for data manipulation, whether you are formatting a single string, ranking a list of products, or summarizing total sales.



# Using Scalar Functions

## What are Scalar Functions?
**Scalar functions** operate on a single row of data and return exactly **one single value**. 
- They can take zero inputs (e.g., `GETDATE()`), one input (e.g., `UPPER()`), or multiple inputs (e.g., `ROUND()`).
- Because they return a single value, they can be used anywhere a single value is expected: in `SELECT` lists, `WHERE` clause predicates, or the `SET` clause of an `UPDATE` statement.

## Important Considerations
When using scalar functions, keep these two concepts in mind:

### 1. Determinism
- **Deterministic**: Always returns the same output for the same input and database state (e.g., `ROUND(1.1, 0)` always returns `1.0`).
- **Nondeterministic**: Returns different outputs each time it is called (e.g., `GETDATE()` returns the current system time). 
- *Note:* Results from nondeterministic functions **cannot be indexed**, which can limit the query optimizer's ability to create the most efficient execution plan.

### 2. Collation
When manipulating character data, the function must know which **collation** (sort order and case-sensitivity rules) to use. Some functions use the collation of the input value, while others default to the database's collation if none is supplied.

***

## Date and Time Functions

SQL Server provides a rich set of functions to extract, manipulate, and calculate differences between dates and times.

### Visualizing Date Extraction
When you have a single `DATETIME` value, scalar functions allow you to break it down into its individual components.

```mermaid
flowchart TD
    classDef date fill:#e3f2fd,stroke:#1565c0,stroke-width:2px;
    classDef func fill:#fff3e0,stroke:#ef6c00,stroke-width:2px;
    classDef result fill:#e8f5e9,stroke:#2e7d32,stroke-width:2px;

    A["OrderDate: 2008-06-01 00:00:00"]:::date 
    
    A --> B["YEAR(OrderDate)"]:::func --> C["2008"]:::result
    A --> D["DATENAME(mm, OrderDate)"]:::func --> E["June"]:::result
    A --> F["DAY(OrderDate)"]:::func --> G["1"]:::result
    A --> H["DATENAME(dw, OrderDate)"]:::func --> I["Sunday"]:::result
```

***

**Date and Time Examples**
```sql
-- Extracting various parts of a date and calculating time differences
SELECT SalesOrderID,
       OrderDate,
       YEAR(OrderDate) AS OrderYear,
       DATENAME(mm, OrderDate) AS OrderMonth,
       DAY(OrderDate) AS OrderDay,
       DATENAME(dw, OrderDate) AS OrderWeekDay,
       DATEDIFF(yy, OrderDate, GETDATE()) AS YearsSinceOrder
FROM Sales.SalesOrderHeader;
```

***

## Mathematical Functions

Mathematical scalar functions allow you to perform calculations on numeric data directly within your query.

| Function | Description |
| :--- | :--- |
| `ROUND(n, length)` | Rounds a number to a specified length or precision. |
| `FLOOR(n)` | Returns the largest integer less than or equal to the specified value. |
| `CEILING(n)` | Returns the smallest integer greater than or equal to the specified value. |
| `SQUARE(n)` | Returns the square of the specified value. |
| `SQRT(n)` | Returns the square root of the specified value. |
| `LOG(n)` | Returns the natural logarithm of the specified value. |
| `RAND()` | Returns a pseudorandom float value from 0 through 1 (Nondeterministic). |

***

**Mathematical Examples**
```sql
-- Applying various mathematical transformations to a numeric column
SELECT TaxAmt,
       ROUND(TaxAmt, 0) AS Rounded,
       FLOOR(TaxAmt) AS Floor,
       CEILING(TaxAmt) AS Ceiling,
       SQUARE(TaxAmt) AS Squared,
       SQRT(TaxAmt) AS Root,
       LOG(TaxAmt) AS Log,
       TaxAmt * RAND() AS Randomized
FROM Sales.SalesOrderHeader;
```

***

## String Functions

String functions are used to manipulate character data. They are incredibly useful for formatting output, parsing text, or cleaning data.

### Key String Functions:
- **`UPPER()` / `LOWER()`**: Changes the case of the string.
- **`LEN()`**: Returns the number of characters in the string.
- **`REVERSE()`**: Reverses the order of the characters.
- **`CHARINDEX()`**: Finds the starting position of a specified substring (returns an integer).
- **`LEFT()` / `RIGHT()`**: Extracts a specified number of characters from the left or right side.
- **`SUBSTRING()`**: Extracts a specific part of a string based on a start position and length.

***

**String Examples**
```sql
-- Manipulating and parsing company names
SELECT CompanyName,
       UPPER(CompanyName) AS UpperCase,
       LOWER(CompanyName) AS LowerCase,
       LEN(CompanyName) AS Length,
       REVERSE(CompanyName) AS Reversed,
       CHARINDEX(' ', CompanyName) AS FirstSpace,
       LEFT(CompanyName, CHARINDEX(' ', CompanyName)) AS FirstWord,
       SUBSTRING(CompanyName, CHARINDEX(' ', CompanyName) + 1, LEN(CompanyName)) AS RestOfName
FROM Sales.Customer;
```

***

## Logical Functions

Logical functions evaluate an input expression and return an appropriate value based on the result. They are excellent for applying conditional logic directly within a `SELECT` statement without needing complex `CASE` expressions.

### 1. `IIF ( boolean_expression, true_value, false_value )`
Evaluates a Boolean condition. If `TRUE`, it returns the second argument. If `FALSE`, it returns the third argument.

### 2. `CHOOSE ( index, val_1, val_2, ... val_n )`
Evaluates an integer expression (1-based index) and returns the corresponding value from the provided list. 
*(e.g., If the index is 2, it returns `val_2`)*.

```mermaid
flowchart LR
    classDef iif fill:#e3f2fd,stroke:#1565c0,stroke-width:2px;
    classDef choose fill:#fff3e0,stroke:#ef6c00,stroke-width:2px;
    
    subgraph IIF Logic
        I1{"Is AddressType = 'Main Office'?"}:::iif
        I1 -->|Yes| I2["Return 'Billing'"]
        I1 -->|No| I3["Return 'Mailing'"]
    end
    
    subgraph CHOOSE Logic
        C1["Status Integer"]:::choose
        C1 -->|1| C2["'Ordered'"]
        C1 -->|2| C3["'Shipped'"]
        C1 -->|3| C4["'Delivered'"]
    end
```

***

**Logical Function Examples**
```sql
-- ==========================================
-- IIF EXAMPLE
-- ==========================================
-- Maps 'Main Office' to 'Billing', and everything else to 'Mailing'
SELECT AddressType,
       IIF(AddressType = 'Main Office', 'Billing', 'Mailing') AS UseAddressFor
FROM Sales.CustomerAddress;


-- ==========================================
-- CHOOSE EXAMPLE
-- ==========================================
-- Converts numeric Status codes into readable text strings
SELECT SalesOrderID, Status,
       CHOOSE(Status, 'Ordered', 'Shipped', 'Delivered') AS OrderStatus
FROM Sales.SalesOrderHeader;
```

***

## Summary

- **Scalar functions** take one or more inputs and return a **single value**.
- They can be used in `SELECT`, `WHERE`, and `UPDATE` clauses.
- **Deterministic** functions always return the same result for the same input and can be indexed. **Nondeterministic** functions (like `GETDATE()` or `RAND()`) cannot.
- **Date/Time functions** allow you to extract parts of a date or calculate differences.
- **Mathematical functions** perform calculations like rounding, flooring, and square roots.
- **String functions** manipulate text, allowing you to change case, find substrings, and measure length.
- **Logical functions** (`IIF` and `CHOOSE`) provide a clean, shorthand way to apply conditional logic and map values directly in your query output.



# Using Ranking and Rowset Functions

## How They Differ from Scalar Functions
Unlike scalar functions, which operate on a single row and return a single value, **ranking and rowset functions** operate on a **set of rows** as input and return a **set of rows** as output. 

### Input/Output Comparison
```mermaid
flowchart LR
    classDef scalar fill:#e3f2fd,stroke:#1565c0,stroke-width:2px;
    classDef rowset fill:#fff3e0,stroke:#ef6c00,stroke-width:2px;
    
    subgraph Scalar Functions
        S1["Input: 1 Row"]:::scalar --> S2["Output: 1 Value"]:::scalar
    end
    
    subgraph Ranking & Rowset Functions
        R1["Input: Set of Rows"]:::rowset --> R2["Output: Set of Rows"]:::rowset
    end
```

***

## 1. Ranking Functions

Ranking functions allow you to perform calculations against a user-defined set (or window) of rows. Common uses include assigning row numbers, ranking items by price, or dividing data into buckets.

### The `OVER` Clause
To use a ranking function, you must pair it with the **`OVER`** clause. The `OVER` clause defines the "window" of rows the function operates on. It requires an `ORDER BY` clause to determine the sequence of the ranking.

**Basic Syntax:**
```sql
FUNCTION_NAME() OVER (ORDER BY <column> [ASC|DESC])
```

### Example: Ranking by Price
Using the `RANK()` function, we can assign a rank to products based on their `ListPrice`, with the highest price ranked as #1.

***

**Basic Ranking Example**
```sql
-- Calculate a ranking based on ListPrice (highest price = Rank 1)
SELECT TOP 100 ProductID, Name, ListPrice,
       RANK() OVER(ORDER BY ListPrice DESC) AS RankByPrice
FROM Production.Product AS p
ORDER BY RankByPrice;
```

***

## Partitioning Data with `PARTITION BY`

By default, a ranking function treats the entire result set as a single group. However, you often want to reset the ranking for specific categories or groups. 

You can achieve this by adding the **`PARTITION BY`** clause inside the `OVER` clause. This divides the result set into partitions, and the ranking function is applied to each partition independently.

### Visualizing Partitions
```mermaid
flowchart TD
    classDef data fill:#f9f9f9,stroke:#333,stroke-width:2px;
    classDef part fill:#e8f5e9,stroke:#2e7d32,stroke-width:2px;
    classDef rank fill:#fff3e0,stroke:#ef6c00,stroke-width:2px;

    A["All Products"]:::data --> B["Partition 1: Bib-Shorts"]:::part
    A --> C["Partition 2: Bike Racks"]:::part
    A --> D["Partition 3: Bottles and Cages"]:::part
    
    B --> E["Rank 1, 2..."]:::rank
    C --> F["Rank 1..."]:::rank
    D --> G["Rank 1, 2, 3..."]:::rank
    
    note["The rank resets to 1 at the start of each partition!"]
```

***

**Partitioned Ranking Example**
```sql
-- Rank products by price, but reset the rank for EACH Category
SELECT c.Name AS Category, p.Name AS Product, ListPrice,
       RANK() OVER(PARTITION BY c.Name ORDER BY ListPrice DESC) AS RankByPrice
FROM Production.Product AS p
JOIN Production.ProductCategory AS c
    ON p.ProductCategoryID = c.ProductCategoryID
ORDER BY Category, RankByPrice;
```

***

## Handling Ties in Ranking

When using the `RANK()` function, if multiple rows have the exact same value for the `ORDER BY` column, they receive the **same rank**. However, the next rank number will be **skipped**.

### Example of `RANK()` Behavior:
| ListPrice | Rank | Note |
| :--- | :--- | :--- |
| $3578.27 | **1** | Tie |
| $3578.27 | **1** | Tie |
| $3578.27 | **1** | Tie |
| $3399.99 | **4** | *Notice that 2 and 3 are skipped!* |

### Alternative Ranking Functions
Depending on your business requirements, you may want to handle ties differently. T-SQL provides other ranking functions:
- **`DENSE_RANK()`**: Assigns the same rank to ties, but **does not skip** subsequent rank numbers (the next rank would be 2).
- **`ROW_NUMBER()`**: Assigns a unique, sequential integer to each row, even if there are ties (breaks ties arbitrarily based on the `ORDER BY`).
- **`NTILE(n)`**: Distributes the rows into a specified number of roughly equal groups (or "tiles"/"buckets").

***

## 2. Rowset Functions

Unlike ranking functions that add a column to an existing result set, **rowset functions** return an entirely **virtual table**. This virtual table can be used directly in the `FROM` clause as a data source, just like a regular table or view.

### Categories of Rowset Functions:
1. **Remote Data Access**: 
   - `OPENDATASOURCE`, `OPENQUERY`, `OPENROWSET`
   - These allow you to pass a query to a remote database server and return the result rows. *(Note: Requires advanced server options to be enabled).*
2. **Structured Data Parsing**: 
   - `OPENXML`, `OPENJSON`
   - These allow you to query structured data (XML or JSON format) and extract the values into a relational, tabular rowset.

***

**Rowset Function Example**
```sql
-- ==========================================
-- ROWSET FUNCTION EXAMPLE (Conceptual)
-- ==========================================
-- Using OPENROWSET to query a remote SQL Server instance.
-- This returns a virtual table 'a' that can be queried like any normal table.

SELECT a.*
FROM OPENROWSET('SQLNCLI', 'Server=SalesDB;Trusted_Connection=yes;',
    'SELECT Name, ListPrice
     FROM AdventureWorks.Production.Product') AS a;

-- Note: OPENJSON and OPENXML follow a similar pattern, 
-- parsing semi-structured text into relational columns.
```

***

## Summary

- **Ranking and Rowset functions** operate on a **set of rows** and return a **set of rows**, unlike scalar functions.
- **Ranking Functions** (`RANK`, `DENSE_RANK`, `ROW_NUMBER`, `NTILE`) require the **`OVER`** clause to define the window of rows.
- Use **`PARTITION BY`** inside the `OVER` clause to divide the data into groups and reset the ranking for each group.
- `RANK()` skips numbers after a tie. Use `DENSE_RANK()` if you want consecutive numbers, or `ROW_NUMBER()` to force unique sequential numbers.
- **Rowset Functions** (`OPENROWSET`, `OPENJSON`, `OPENXML`) return a virtual table that can be used in the `FROM` clause to query remote servers or parse semi-structured data.

```mermaid
graph TD
    A[(Raw Data: Products & Prices)] --> B[PARTITION BY Category]
    
    B --> C[Category: Bikes]
    B --> D[Category: Accessories]
    
    C --> E[ORDER BY Price DESC]
    D --> F[ORDER BY Price DESC]
    
    E --> G[Rank 1: $3000 <br> Rank 2: $2500 <br> Rank 3: $1000]
    F --> H[Rank 1: $120 <br> Rank 2: $50 <br> Rank 2: $50 <br> Rank 4: $10]
    
    style B fill:#e1f5fe,stroke:#0288d1
    style G fill:#d4edda,stroke:#2e7d32
    style H fill:#d4edda,stroke:#2e7d32
```

# Using Aggregate Functions

## Overview
Aggregate functions perform calculations across a set of values and return a **single summarizing result**. Unlike scalar functions that operate row-by-row, aggregate functions collapse a set of rows into a single value (e.g., calculating the total sales or the average price).

## Crucial Rules for Aggregate Functions
When working with aggregate functions, you must adhere to the following rules:

1. **Return Type**: They return a single (scalar) value.
2. **Placement**: They can be used in the `SELECT`, `HAVING`, and `ORDER BY` clauses. However, they **cannot** be used in the `WHERE` clause (because `WHERE` filters rows *before* aggregation).
3. **NULL Handling**: Aggregate functions **ignore NULLs**, with the sole exception of `COUNT(*)`.
4. **Aliases**: Aggregate functions in a `SELECT` list do not generate a column header automatically. Always use `AS` to provide an alias.
5. **Scope**: If there is no `GROUP BY` clause, the aggregate operates on **all** rows satisfying the `WHERE` clause.
6. **The Golden Rule**: You **cannot** combine aggregate functions with non-aggregated columns in the same `SELECT` list unless you use a `GROUP BY` clause.

***

## Common Built-in Aggregate Functions

| Function | Syntax | Description |
| :--- | :--- | :--- |
| **`SUM`** | `SUM(expression)` | Totals all the non-NULL numeric values in a column. |
| **`AVG`** | `AVG(expression)` | Averages all the non-NULL numeric values (Sum / Count). |
| **`MIN`** | `MIN(expression)` | Returns the smallest number, earliest date/time, or first-occurring string (alphabetical). |
| **`MAX`** | `MAX(expression)` | Returns the largest number, latest date/time, or last-occurring string (alphabetical). |
| **`COUNT`** | `COUNT(*)` or `COUNT(expr)` | `COUNT(*)` counts **all** rows (including NULLs). `COUNT(expr)` counts only **non-NULL** rows. Returns an `int`. |
| **`COUNT_BIG`**| `COUNT_BIG(expr)` | Same as `COUNT`, but returns a `bigint` (used for massive datasets exceeding 2 billion rows). |

***

**Basic Aggregate Examples**
```sql
-- Calculating average, minimum, and maximum prices for ALL products
SELECT AVG(ListPrice) AS AveragePrice,
       MIN(ListPrice) AS MinimumPrice,
       MAX(ListPrice) AS MaximumPrice
FROM Production.Product;

-- Filtering the dataset BEFORE aggregation using WHERE
-- This calculates the aggregates ONLY for products in Category 15
SELECT AVG(ListPrice) AS AveragePrice,
       MIN(ListPrice) AS MinimumPrice,
       MAX(ListPrice) AS MaximumPrice
FROM Production.Product
WHERE ProductCategoryID = 15;
```

***

## The Golden Rule of Aggregation

When you use an aggregate function, SQL Server collapses multiple rows into a single row. If you try to include a standard, non-aggregated column in the same `SELECT` list, SQL Server won't know which row's value to display for that column, resulting in **Error 8120**.

### Visualizing the Logical Conflict
```mermaid
flowchart TD
    classDef error fill:#ffebee,stroke:#c62828,stroke-width:2px;
    classDef question fill:#fff3e0,stroke:#ef6c00,stroke-width:2px;
    
    A["SELECT ProductCategoryID, AVG(ListPrice)"] --> B["SQL Server calculates ONE average price for the whole table"]
    B --> C["But which ProductCategoryID should it display on that single row?"]:::question
    C --> D["❌ ERROR 8120: Column is invalid because it's not in an aggregate or GROUP BY"]:::error
    
    note["Solution: Use GROUP BY ProductCategoryID to calculate an average FOR EACH category."]
```

***

**Demonstrating the Error**
```sql
-- ❌ THIS WILL FAIL WITH ERROR 8120
-- You cannot mix a raw column with an aggregate without GROUP BY
SELECT ProductCategoryID, 
       AVG(ListPrice) AS AveragePrice,
       MIN(ListPrice) AS MinimumPrice,
       MAX(ListPrice) AS MaximumPrice
FROM Production.Product;
```

***

## Aggregates on Non-Numeric Data & Nesting Functions

### Data Type Restrictions
- **`SUM` and `AVG`** can **only** be used on numeric data types (int, money, float, decimal).
- **`MIN` and `MAX`** are highly versatile. They can be used on numeric data, dates/times (returning earliest/latest), and strings (returning first/last based on collation/alphabetical order).

### Nesting Functions
You can nest scalar functions inside aggregate functions. The scalar function is evaluated first for each row, and then the aggregate function processes those results.

***

**Non-Numeric and Nested Examples**
```sql
-- 1. Using MIN and MAX on Strings (Alphabetical order)
SELECT MIN(CompanyName) AS MinCustomer, 
       MAX(CompanyName) AS MaxCustomer
FROM SalesLT.Customer;

-- 2. Nesting a Scalar Function (YEAR) inside an Aggregate (MIN/MAX)
-- Extracts the year first, then finds the earliest and latest year
SELECT MIN(YEAR(OrderDate)) AS EarliestYear,
       MAX(YEAR(OrderDate)) AS LatestYear
FROM Sales.SalesOrderHeader;
```

***

## Using `DISTINCT` with Aggregate Functions

When you place the `DISTINCT` keyword inside an aggregate function, it removes duplicate values from the input column **before** computing the summary value. This is incredibly useful for counting unique occurrences.

### Comparison of COUNT variations:
- `COUNT(*)`: Counts every single row in the table.
- `COUNT(column)`: Counts rows where the column is not NULL.
- `COUNT(DISTINCT column)`: Counts only the **unique, non-NULL** values in the column.

***

**DISTINCT Examples**
```sql
-- Counting total orders placed
SELECT COUNT(*) AS TotalOrders FROM Sales.SalesOrderHeader;

-- Counting how many orders have a CustomerID (ignores NULL CustomerIDs)
SELECT COUNT(CustomerID) AS OrdersWithCustomer FROM Sales.SalesOrderHeader;

-- Counting the number of UNIQUE customers who have placed orders
-- (If a customer placed 5 orders, they are only counted once)
SELECT COUNT(DISTINCT CustomerID) AS UniqueCustomers FROM Sales.SalesOrderHeader;
```

***

## The NULL Trap in Aggregates

Because aggregate functions ignore `NULL` values, calculating an average manually can yield drastically different results than using the built-in `AVG()` function if your data contains `NULL`s.

### How `AVG()` calculates vs. Manual Calculation
```mermaid
flowchart TD
    classDef avg fill:#e3f2fd,stroke:#1565c0,stroke-width:2px;
    classDef manual fill:#fff3e0,stroke:#ef6c00,stroke-width:2px;
    classDef data fill:#f9f9f9,stroke:#333,stroke-width:2px;

    A["Data: 10, 20, NULL, 40"]:::data
    
    subgraph Built-in AVG
        B1["SUM = 70"]:::avg
        B2["COUNT(non-NULL) = 3"]:::avg
        B3["70 / 3 = 23.33"]:::avg
    end
    
    subgraph Manual Average SUM/COUNT*
        C1["SUM = 70"]:::manual
        C2["COUNT(*) all rows = 4"]:::manual
        C3["70 / 4 = 17.50"]:::manual
    end
    
    A --> B1 & C1
    B1 --> B2 --> B3
    C1 --> C2 --> C3
```

### The Solution: `COALESCE`
If your business logic requires treating `NULL` as `0` so it is included in the denominator of an average, use the `COALESCE()` function to replace `NULL`s before aggregating: `AVG(COALESCE(Column, 0))`.

***

**Demonstrating the NULL Trap**
```sql
-- Creating a temporary table to demonstrate the NULL trap
CREATE TABLE #t1 (C1 int, C2 int);
INSERT INTO #t1 VALUES (1, NULL), (2, 10), (3, 20), (4, 30), (5, 40), (6, 50);

-- Comparing built-in AVG vs Manual Calculation
SELECT SUM(C2) AS sum_nonnulls, 
       COUNT(*) AS count_all_rows, 
       COUNT(C2) AS count_nonnulls, 
       AVG(C2) AS average_builtin,         -- Divides by 5 (ignores NULL) -> 30
       (SUM(C2)/COUNT(*)) AS arith_average -- Divides by 6 (includes NULL row) -> 25
FROM #t1;

-- Cleaning up
DROP TABLE #t1;
```

***

## Summary

- **Aggregate functions** (`SUM`, `AVG`, `MIN`, `MAX`, `COUNT`) collapse multiple rows into a single summary value.
- **The Golden Rule**: You cannot mix aggregated and non-aggregated columns in a `SELECT` list without using `GROUP BY`.
- **NULL Handling**: Aggregates ignore `NULL`s. `COUNT(*)` is the only exception that counts all rows regardless of `NULL`s.
- **Data Types**: `SUM` and `AVG` are for numbers only. `MIN` and `MAX` work on numbers, dates, and strings.
- **`DISTINCT`**: Use `COUNT(DISTINCT column)` to count unique values.
- **The NULL Trap**: `AVG(column)` divides by the count of *non-NULL* rows. If you need to include `NULL`s as zeros in your average, use `AVG(COALESCE(column, 0))`.



# Summarizing Data with `GROUP BY`

## Subdividing Data into Groups
While aggregate functions (like `SUM` or `COUNT`) are useful for summarizing an entire table into a single row, you often need to arrange your data into subsets *before* summarizing it. 

The **`GROUP BY`** clause allows you to subdivide the contents of the virtual table (created by the `FROM` and `WHERE` clauses) into distinct groups of rows. The aggregate functions are then applied to each group independently.

### Visualizing the GROUP BY Process
When you group by a column, SQL Server collapses all rows that share the same value in that column into a single output row.

```mermaid
flowchart TD
    classDef raw fill:#f9f9f9,stroke:#333,stroke-width:2px;
    classDef group fill:#e3f2fd,stroke:#1565c0,stroke-width:2px;
    classDef result fill:#e8f5e9,stroke:#2e7d32,stroke-width:2px;

    subgraph Raw Data [Raw Rows in Virtual Table]
        R1["CustID: 1234 | Order: A"]:::raw
        R2["CustID: 1234 | Order: B"]:::raw
        R3["CustID: 1234 | Order: C"]:::raw
        R4["CustID: 1005 | Order: D"]:::raw
    end

    subgraph Grouping [GROUP BY CustomerID]
        G1["Group 1234\n(3 rows)"]:::group
        G2["Group 1005\n(1 row)"]:::group
    end

    subgraph Result [Final Output]
        O1["CustID: 1234 | Count: 3"]:::result
        O2["CustID: 1005 | Count: 1"]:::result
    end

    R1 & R2 & R3 --> G1
    R4 --> G2
    G1 --> O1
    G2 --> O2
```

***

## `GROUP BY` vs. `DISTINCT`

If you only want to find the unique values in a column, a `GROUP BY` clause without any aggregate functions is logically equivalent to using the `DISTINCT` keyword.

```sql
-- These two queries return the exact same result:
SELECT CustomerID FROM Sales.SalesOrderHeader GROUP BY CustomerID;
SELECT DISTINCT CustomerID FROM Sales.SalesOrderHeader;
```

### The Power of GROUP BY
The real power of `GROUP BY` is revealed when you add aggregate functions to the `SELECT` list. While `DISTINCT` just removes duplicates, `GROUP BY` allows you to calculate summaries (like counts, sums, or averages) **for each distinct group**.

***

**Basic GROUP BY and COUNT**
```sql
-- Counting the number of orders placed by EACH customer
-- The raw table has 830 rows, but this groups them into 89 distinct Customer groups.

SELECT CustomerID, COUNT(*) AS OrderCount
FROM Sales.SalesOrderHeader
GROUP BY CustomerID;

-- ==========================================
-- Sorting GROUP BY Results
-- ==========================================
-- GROUP BY does NOT guarantee the order of the results.
-- If you need a specific order, you MUST use an explicit ORDER BY clause.

SELECT CustomerID, COUNT(*) AS OrderCount
FROM Sales.SalesOrderHeader
GROUP BY CustomerID
ORDER BY CustomerID; -- Now sorted in ascending order of CustomerID
```

***

## Logical Processing Order and Column Aliases

To understand how `GROUP BY` works, it is crucial to remember the logical order in which SQL Server processes a `SELECT` statement:

```mermaid
graph TD
    classDef step fill:#f9f9f9,stroke:#333,stroke-width:2px;
    classDef highlight fill:#fff3e0,stroke:#ef6c00,stroke-width:2px;

    A["1. FROM"]:::step --> B["2. WHERE"]:::step
    B --> C["3. GROUP BY"]:::highlight
    C --> D["4. HAVING"]:::step
    D --> E["5. SELECT (Aliases created here)"]:::highlight
    E --> F["6. ORDER BY"]:::highlight
```

### The Rule for Column Aliases
Because column aliases are assigned during the **`SELECT`** phase (Step 5), they do not exist yet when the **`GROUP BY`** phase (Step 3) is evaluated. 

- ❌ **You CANNOT use an alias in the `GROUP BY` clause.** (It will throw an "Invalid column name" error).
- ✅ **You CAN use an alias in the `ORDER BY` clause.** (Because `ORDER BY` happens after `SELECT`).

***

**Aliases in GROUP BY vs ORDER BY**
```sql
-- ❌ THIS WILL FAIL
-- The alias 'Customer' doesn't exist yet when GROUP BY is processed.
SELECT CustomerID AS Customer, COUNT(*) AS OrderCount
FROM Sales.SalesOrderHeader
GROUP BY Customer 
ORDER BY Customer;

-- ✅ THIS WILL SUCCEED
-- Use the actual column name in GROUP BY, but you can use the alias in ORDER BY.
SELECT CustomerID AS Customer, COUNT(*) AS OrderCount
FROM Sales.SalesOrderHeader
GROUP BY CustomerID 
ORDER BY Customer;
```

***

## Troubleshooting Error 8120: The Golden Rule

A common obstacle when learning `GROUP BY` is encountering this error:
> *Msg 8120: Column '<column_name>' is invalid in the select list because it is not contained in either an aggregate function or the GROUP BY clause.*

### The Logic Behind the Error
When you use `GROUP BY`, SQL Server collapses multiple rows into a single row per group. 
If you ask for a column in the `SELECT` list that is **not** in the `GROUP BY` clause, SQL Server won't know which of the original rows' values to display for that column. 

### The Golden Rule of GROUP BY
Every column in your `SELECT` list **must** satisfy one of two conditions:
1. It is included in the **`GROUP BY`** clause.
2. It is wrapped inside an **aggregate function** (like `SUM()`, `MAX()`, `COUNT()`).

***

**Demonstrating and Fixing Error 8120**
```sql
-- ❌ THIS WILL FAIL WITH ERROR 8120
-- We are grouping by CustomerID (creating one row per customer).
-- But a customer can have multiple different PurchaseOrderNumbers. 
-- Which PurchaseOrderNumber should SQL Server display on that single row? It doesn't know!
SELECT CustomerID, PurchaseOrderNumber, COUNT(*) AS OrderCount
FROM Sales.SalesOrderHeader
GROUP BY CustomerID;


-- ✅ FIX 1: Add the column to the GROUP BY clause
-- Now we get one row for each unique combination of Customer AND PurchaseOrder.
SELECT CustomerID, PurchaseOrderNumber, COUNT(*) AS OrderCount
FROM Sales.SalesOrderHeader
GROUP BY CustomerID, PurchaseOrderNumber;


-- ✅ FIX 2: Wrap the column in an aggregate function
-- If we just want to see ONE of the purchase orders per customer (e.g., the last one alphabetically).
SELECT CustomerID, MAX(PurchaseOrderNumber) AS LastPurchaseOrder, COUNT(*) AS OrderCount
FROM Sales.SalesOrderHeader
GROUP BY CustomerID;
```

***

## Summary

- **`GROUP BY`** subdivides the result set into groups based on one or more columns, allowing you to perform aggregations on each group independently.
- `GROUP BY` without aggregates is logically equivalent to `DISTINCT`, but `GROUP BY` allows you to add summary calculations.
- **Logical Processing**: `GROUP BY` happens *before* `SELECT`. Therefore, you **cannot** use column aliases in the `GROUP BY` clause, but you **can** use them in the `ORDER BY` clause.
- **Sorting**: `GROUP BY` does not guarantee output order. Always use an explicit `ORDER BY` if sorting is required.
- **The Golden Rule (Error 8120)**: Every column in the `SELECT` list must either be in the `GROUP BY` clause or be wrapped in an aggregate function.
- You can group by **multiple columns** to create groups based on unique combinations of those columns.

```mermaid
graph LR
    subgraph Virtual_Table_For_Raw_Data["Virtual Table: Raw Data"]
    A[Order 1: Cust 1234]
    B[Order 2: Cust 1005]
    C[Order 3: Cust 1234]
    D[Order 4: Cust 1234]
    end

    subgraph GROUP_BY_CustomerID["GROUP BY CustomerID"]
    E[Bucket: 1234]
    F[Bucket: 1005]
    end

    subgraph Aggregation_Phase["Aggregation Phase"]
    G([COUNT = 3])
    H([COUNT = 1])
    end

    A --> E
    B --> F
    C --> E
    D --> E

    E --> G
    F --> H
    
    style E fill:#e1f5fe,stroke:#0288d1
    style F fill:#e1f5fe,stroke:#0288d1
    style G fill:#d4edda,stroke:#2e7d32
    style H fill:#d4edda,stroke:#2e7d32
```

# Filtering Groups with `HAVING`

## What is the HAVING Clause?
Once you have created groups using a `GROUP BY` clause, you often need to filter those groups based on the results of an aggregate function. The **`HAVING`** clause acts as a filter specifically for groups.

Think of it as a `WHERE` clause, but one that operates **after** the data has been grouped and aggregated.

### Basic Syntax
```sql
SELECT <columns>, AGGREGATE(<column>) AS alias
FROM <table>
GROUP BY <columns>
HAVING AGGREGATE(<column>) <comparison> <value>;
```

***

## Comparing `WHERE` and `HAVING`

While both clauses filter data, they operate at completely different stages of query processing and serve different purposes.

### The Key Distinction:
- **`WHERE`** filters individual **rows** *before* any groups are formed. It operates on the raw data returned by the `FROM` clause.
- **`HAVING`** filters entire **groups** *after* the `GROUP BY` clause has been processed. It typically evaluates the results of aggregate functions.

### Visualizing the Difference
```mermaid
flowchart TD
    classDef raw fill:#f9f9f9,stroke:#333,stroke-width:2px;
    classDef where fill:#ffebee,stroke:#c62828,stroke-width:2px;
    classDef group fill:#e3f2fd,stroke:#1565c0,stroke-width:2px;
    classDef having fill:#fff3e0,stroke:#ef6c00,stroke-width:2px;
    classDef result fill:#e8f5e9,stroke:#2e7d32,stroke-width:2px;

    A["1. FROM Clause\n(Raw rows from tables)"]:::raw --> B["2. WHERE Clause\nFilters individual ROWS\n(e.g., WHERE OrderDate > '2023-01-01')"]:::where
    B --> C["3. GROUP BY Clause\nForms groups from remaining rows"]:::group
    C --> D["4. Aggregate Functions\nCalculate SUM, COUNT, AVG per group"]:::group
    D --> E["5. HAVING Clause\nFilters entire GROUPS\n(e.g., HAVING COUNT(*) > 10)"]:::having
    E --> F["6. Final Result Set\nOnly groups meeting the HAVING condition"]:::result
```

### When to Use Which?
| Filter Type | Operates On | When to Use | Example |
| :--- | :--- | :--- | :--- |
| **`WHERE`** | Individual Rows | Filter raw data before grouping | `WHERE OrderDate > '2023-01-01'` |
| **`HAVING`** | Groups / Aggregates | Filter based on summary calculations | `HAVING COUNT(*) > 10` |

***

**Basic HAVING Example**
```sql
-- ==========================================
-- USING HAVING TO FILTER GROUPS
-- ==========================================
-- Count the number of orders for each customer,
-- but ONLY return customers who have placed MORE THAN 10 orders.

SELECT CustomerID,
       COUNT(*) AS OrderCount
FROM Sales.SalesOrderHeader
GROUP BY CustomerID
HAVING COUNT(*) > 10;
```

***

## Combining `WHERE` and `HAVING` in a Single Query

You can use both `WHERE` and `HAVING` in the same query. When you do, they work in sequence:

1. **`WHERE`** runs first, filtering out individual rows that don't meet the condition.
2. **`GROUP BY`** forms groups from the *remaining* rows.
3. **`HAVING`** runs last, filtering out entire groups that don't meet the aggregate condition.

### Scenario Example
Imagine you want to find customers who placed more than 5 orders **in the year 2008**:
- Use `WHERE` to limit the rows to only 2008 orders.
- Use `GROUP BY` to group by customer.
- Use `HAVING` to keep only customers with more than 5 orders in that year.

```mermaid
sequenceDiagram
    participant FROM as FROM Clause
    participant WHERE as WHERE Clause
    participant GROUP as GROUP BY
    participant HAVING as HAVING Clause
    
    FROM->>WHERE: All 10,000 order rows
    WHERE->>WHERE: Filter: OrderDate in 2008
    WHERE->>GROUP: 3,000 rows remaining
    GROUP->>GROUP: Group by CustomerID
    GROUP->>HAVING: 200 customer groups
    HAVING->>HAVING: Filter: COUNT(*) > 5
    HAVING->>HAVING: 45 groups remain
```

***

**Combining WHERE and HAVING**
```sql
-- ==========================================
-- COMBINING WHERE AND HAVING
-- ==========================================
-- Find customers who placed more than 5 orders in 2008.
-- WHERE filters rows first, then HAVING filters the groups.

SELECT CustomerID,
       COUNT(*) AS OrderCount
FROM Sales.SalesOrderHeader
WHERE YEAR(OrderDate) = 2008  -- Step 1: Keep only 2008 orders
GROUP BY CustomerID            -- Step 2: Group remaining rows by customer
HAVING COUNT(*) > 5            -- Step 3: Keep only groups with more than 5 orders
ORDER BY OrderCount DESC;
```

***

## ⚠️ Common Mistakes with HAVING

### Mistake 1: Using HAVING to filter non-aggregated data
If you are filtering on a raw column value (not an aggregate), you should use `WHERE`, not `HAVING`. While SQL Server *might* allow it in some cases, it is semantically incorrect and can lead to poor performance.

```sql
-- ❌ Inefficient: Using HAVING to filter a raw column
SELECT CustomerID, COUNT(*) AS OrderCount
FROM Sales.SalesOrderHeader
GROUP BY CustomerID
HAVING CustomerID = 1005;  -- Should be in WHERE!

-- ✅ Correct: Use WHERE for row-level filtering
SELECT CustomerID, COUNT(*) AS OrderCount
FROM Sales.SalesOrderHeader
WHERE CustomerID = 1005    -- Filters rows BEFORE grouping
GROUP BY CustomerID;
```

### Mistake 2: Using aggregate functions in WHERE
You **cannot** use aggregate functions in a `WHERE` clause because `WHERE` is processed *before* `GROUP BY` and aggregation occurs.

```sql
-- ❌ THIS WILL FAIL
SELECT CustomerID, COUNT(*) AS OrderCount
FROM Sales.SalesOrderHeader
WHERE COUNT(*) > 10  -- ERROR: Aggregates not allowed in WHERE
GROUP BY CustomerID;

-- ✅ Use HAVING for aggregate conditions
SELECT CustomerID, COUNT(*) AS OrderCount
FROM Sales.SalesOrderHeader
GROUP BY CustomerID
HAVING COUNT(*) > 10;  -- Correct!
```

### Quick Decision Guide
```mermaid
flowchart TD
    classDef yes fill:#c8e6c9,stroke:#2e7d32,stroke-width:2px;
    classDef no fill:#ffcdd2,stroke:#c62828,stroke-width:2px;
    classDef decision fill:#fff3e0,stroke:#ef6c00,stroke-width:2px;

    A["Need to filter data?"] --> B{"Are you filtering on an aggregate result?\n(e.g., COUNT, SUM, AVG)"}:::decision
    B -->|Yes| C["Use HAVING"]:::yes
    B -->|No| D{"Are you filtering on a raw column value?\n(e.g., Date, ID, Name)"}:::decision
    D -->|Yes| E["Use WHERE"]:::yes
    D -->|No| F["Review your logic"]:::no
```

***

## Summary

- **`HAVING`** filters **groups** after they have been created by `GROUP BY`.
- **`WHERE`** filters **individual rows** before any grouping occurs.
- **Rule of Thumb**: If your filter condition involves an **aggregate function** (`COUNT`, `SUM`, `AVG`, `MIN`, `MAX`), it must go in the **`HAVING`** clause. If it involves a raw column value, it belongs in the **`WHERE`** clause.
- `WHERE` and `HAVING` can be used together in the same query. `WHERE` runs first, reducing the rows that get grouped, and `HAVING` runs after, filtering the resulting groups.
- Using `HAVING` to filter on non-aggregated columns is a common mistake. Always push row-level filters into the `WHERE` clause for better readability and performance.

```mermaid
graph TD
    A[(Raw Data / FROM)] --> B{1. WHERE Clause}
    B -->|Filters Individual Rows| C[2. GROUP BY Clause]
    C -->|Organizes remaining rows into Buckets| D{3. HAVING Clause}
    D -->|Filters entire Buckets/Aggregations| E[4. SELECT]
    
    style B fill:#f9d0c4,stroke:#c62828,stroke-width:2px
    style C fill:#e1f5fe,stroke:#0288d1,stroke-width:2px
    style D fill:#f3e5f5,stroke:#8e24aa,stroke-width:2px
    style E fill:#d4edda,stroke:#2e7d32,stroke-width:2px
```